# Procesamiento de CSV - Unión de Datasets de Inmuebles

Este notebook une el dataset principal de inmuebles con datos macro de cada país como PIB, desigualdad, población y crimen.

**Objetivo:** Enriquecer la información de cada propiedad con contexto demográfico, económico y de seguridad del país.

In [2]:
import pandas as pd
import os

**Importaciones:**
- `pandas`: librería para leer y manipular datos en formato CSV y tablas
- `os`: librería para trabajar con rutas de archivos en el sistema

In [3]:
COUNTRY_MAP = {
    'united states of america': 'united states',
    'usa': 'united states',
    'us': 'united states',
    'uae': 'united arab emirates',
    'north macedonia': 'north macedonia',
    'macedonia fyrom': 'north macedonia',
    'korea south': 'south korea',
    'south korea': 'south korea',
    'korea north': 'north korea',
    'ivory coast': 'cote divoire',
    'cote divoire': 'cote divoire',
    'viet nam': 'vietnam',
    'lao pdr': 'laos',
    'bolivia plurinational state of': 'bolivia',
    'venezuela bolivarian republic of': 'venezuela',
    'united kingdom': 'united kingdom',
    'uk': 'united kingdom',
    'russian federation': 'russia',
    'czechia': 'czech republic',
    'brunei darussalam': 'brunei',
    'iran islamic republic of': 'iran',
    'palestinian territories': 'palestine',
    'north cyprus': 'northern cyprus',
}

**Diccionario de normalización de países:**

Este diccionario traduce nombres de países con variaciones a un nombre estándar. Por ejemplo:
- 'USA' se normaliza a 'united states'
- 'UAE' se normaliza a 'united arab emirates'

Esto es necesario porque cada archivo CSV usa nombres diferentes para el mismo país.

In [4]:
def clean_country(name):
    if not isinstance(name, str):
        return ''
    
    value = name.strip().lower()
    value = value.replace("'", "")
    value = value.replace('`', '')
    value = value.replace('(', ' ')
    value = value.replace(')', ' ')
    value = value.replace('[', ' ')
    value = value.replace(']', ' ')
    value = value.replace(',', ' ')
    value = ' '.join(value.split())
    
    return COUNTRY_MAP.get(value, value)

**Función `clean_country()`:**

Esta función limpia un nombre de país:
1. Convierte a minúsculas
2. Elimina caracteres especiales como paréntesis, corchetes, comas
3. Busca en el diccionario si existe un nombre normalizado
4. Si no encuentra, devuelve el nombre limpio tal cual

Ejemplo: "United States (USA)" → "united states"

In [6]:
def read_country_file(path, country_col, keep_cols):
    with open(path, encoding='utf-8', errors='replace') as fh:
        df = pd.read_csv(fh)
    
    df['country_norm'] = df[country_col].map(clean_country)
    keep = [country_col] + keep_cols + ['country_norm']
    
    return df[keep]

**Función `read_country_file()`:**

Lee un archivo CSV y normaliza los nombres de país:
- `path`: ruta del archivo CSV
- `country_col`: nombre de la columna que contiene el país
- `keep_cols`: list de columnas que queremos guardar

Pasos:
1. Abre y lee el CSV
2. Crea una nueva columna `country_norm` aplicando `clean_country()` a cada fila
3. Devuelve solo las columnas que nos interesan

In [7]:
with open('../../Data/Raw/world_real_estate_data(147k).csv', encoding='utf-8', errors='replace') as fh:
    real_estate = pd.read_csv(fh)

real_estate['country_norm'] = real_estate['country'].map(clean_country)

print('Dataset de inmuebles cargado:')
print('Filas:', len(real_estate))
print('Países únicos:', real_estate['country_norm'].nunique())
print('Columnas:', real_estate.columns.tolist())

Dataset de inmuebles cargado:
Filas: 147536
Países únicos: 28
Columnas: ['title', 'country', 'location', 'building_construction_year', 'building_total_floors', 'apartment_floor', 'apartment_rooms', 'apartment_bedrooms', 'apartment_bathrooms', 'apartment_total_area', 'apartment_living_area', 'price_in_USD', 'image', 'url', 'country_norm']


**Carga del dataset principal:**

Leemos el archivo de inmuebles desde la carpeta `Raw`:
- 147,536 propiedades
- 28 países
- Información: ubicación, precio, características, construcción, etc.

Se crea la columna `country_norm` para poder unir después con otros datasets.

In [8]:
gdp = read_country_file('../../Data/Raw/countries_by_gpd_(ppp).csv', 'Country/Territory', ['(millions of current Int$)'])
gdp = gdp.rename(columns={'(millions of current Int$)': 'gdp_ppp'})

print('Dataset GDP cargado:', len(gdp), 'países')
print(gdp.head())

Dataset GDP cargado: 197 países
     Country/Territory     gdp_ppp        country_norm
0                China  23,460,170               china
1        United States  21,427,700       united states
2  European Union[n 2]  20,794,743  european union n 2
3                India   9,611,679               india
4                Japan   5,459,155               japan


**Carga de datos de PIB (GDP):**

- `gdp_ppp`: PIB ajustado por paridad de poder adquisitivo
- Se renombra la columna para que sea más clara
- 197 países

In [9]:
gini = read_country_file('../../Data/Raw/countries_by_income_equality.csv', 'Country', ['World Bank\ufffdGini', 'Year'])
gini = gini.rename(columns={'World Bank\ufffdGini': 'gini', 'Year': 'gini_year'})

print('Dataset Gini cargado:', len(gini), 'países')
print(gini.head())

Dataset Gini cargado: 177 países
       Country  gini  gini_year country_norm
0  Afghanistan   NaN        NaN  afghanistan
1      Albania  33.2     2017.0      albania
2      Algeria  27.6     2011.0      algeria
3       Angola  51.3     2018.0       angola
4    Argentina  41.4     2018.0    argentina


**Carga de datos de desigualdad (Gini):**

- `gini`: Coeficiente de Gini (0-100, más alto = más desigualdad)
- `gini_year`: Año de los datos
- 177 países

In [10]:
pop = read_country_file('../../Data/Raw/countries_by_population_(united nations).csv', 'Country/Territory', ['UN continental', 'UN statistical', 'Population (2018)', 'Population (2019)', 'Change'])
pop = pop.rename(columns={
    'UN continental': 'un_continent',
    'UN statistical': 'un_statistical',
    'Population (2018)': 'population_2018',
    'Population (2019)': 'population_2019',
    'Change': 'population_change',
})

print('Dataset Población cargado:', len(pop), 'países')
print(pop.head())

Dataset Población cargado: 233 países
  Country/Territory un_continent      un_statistical population_2018  \
0             China         Asia        Eastern Asia   1,427,647,786   
1             India         Asia       Southern Asia   1,352,642,280   
2     United States     Americas    Northern America     327,096,265   
3         Indonesia         Asia  South-eastern Asia     267,670,543   
4          Pakistan         Asia       Southern Asia     212,228,286   

  population_2019 population_change   country_norm  
0   1,433,783,686             0.43%          china  
1   1,366,417,754             1.02%          india  
2     329,064,917             0.60%  united states  
3     270,625,568             1.10%      indonesia  
4     216,565,318             2.04%       pakistan  


**Carga de datos de población:**

- `un_continent`: Continente según ONU
- `population_2018` y `population_2019`: Población en esos años
- `population_change`: Porcentaje de cambio año a año
- 233 países/territorios

In [11]:
crime = read_country_file('../../Data/Raw/crime_index_by_country.csv', 'Country', ['Crime Index', 'Safety Index'])
crime = crime.rename(columns={'Crime Index': 'crime_index', 'Safety Index': 'safety_index'})

print('Dataset Crimen cargado:', len(crime), 'países')
print(crime.head())

Dataset Crimen cargado: 133 países
            Country  crime_index  safety_index      country_norm
0         Venezuela        84.36         15.64         venezuela
1  Papua New Guinea        80.04         19.96  papua new guinea
2      South Africa        77.29         22.71      south africa
3       Afghanistan        76.97         23.03       afghanistan
4          Honduras        76.65         23.35          honduras


**Carga de datos de seguridad:**

- `crime_index`: Índice de crimen (0-100, más alto = más crimen)
- `safety_index`: Índice de seguridad (100 - crime_index)
- 133 países

In [12]:
unemp = read_country_file('../../Data/Raw/unemployment_rate_by_country.csv', 'Country', ['Last', 'Previous', 'Reference', 'Unit'])
unemp = unemp.rename(columns={
    'Last': 'unemployment_last',
    'Previous': 'unemployment_previous',
    'Reference': 'unemployment_reference',
    'Unit': 'unemployment_unit',
})

print('Dataset Desempleo cargado:', len(unemp), 'países')
print(unemp.head())

Dataset Desempleo cargado: 181 países
                  Country  unemployment_last  unemployment_previous  \
0  Bosnia and Herzegovina              34.85                  34.67   
1                  Angola              34.00                  32.70   
2                 Namibia              33.40                  34.00   
3               Palestine              28.50                  26.50   
4                 Nigeria              27.10                  23.10   

  unemployment_reference unemployment_unit            country_norm  
0                 20-Aug                 %  bosnia and herzegovina  
1                 20-Sep                 %                  angola  
2                 18-Dec                 %                 namibia  
3                 20-Sep                 %               palestine  
4                 20-Jun                 %                 nigeria  


**Carga de datos de desempleo:**

- `unemployment_last`: Tasa de desempleo actual
- `unemployment_previous`: Tasa anterior
- `unemployment_reference`: Fecha de referencia
- 181 países

In [13]:
lat = read_country_file('../../Data/Raw/countries_by_latitude.csv', 'name', ['country', 'latitude', 'longitude'])
lat = lat.rename(columns={'country': 'country_code', 'name': 'country_name'})

print('Dataset Ubicación cargado:', len(lat), 'países')
print(lat.head())

Dataset Ubicación cargado: 245 países
           country_name country_code   latitude  longitude  \
0               Andorra           AD  42.546245   1.601554   
1  United Arab Emirates           AE  23.424076  53.847818   
2           Afghanistan           AF  33.939110  67.709953   
3   Antigua and Barbuda           AG  17.060816 -61.796428   
4              Anguilla           AI  18.220554 -63.068615   

           country_norm  
0               andorra  
1  united arab emirates  
2           afghanistan  
3   antigua and barbuda  
4              anguilla  


**Carga de datos geográficos:**

- `country_code`: Código ISO de 2 letras (p.ej., 'ES' para España)
- `latitude` y `longitude`: Coordenadas
- 245 países/territorios

Estos datos permitirán hacer mapas interactivos más tarde.

In [14]:
codes = read_country_file('../../Data/Raw/plotly_countries_and_codes.csv', 'COUNTRY', ['CODE'])
codes = codes.rename(columns={'CODE': 'country_iso_code'})

print('Dataset Códigos cargado:', len(codes), 'países')
print(codes.head())

Dataset Códigos cargado: 222 países
          COUNTRY country_iso_code    country_norm
0     Afghanistan              AFG     afghanistan
1         Albania              ALB         albania
2         Algeria              DZA         algeria
3  American Samoa              ASM  american samoa
4         Andorra              AND         andorra


**Carga de códigos ISO de 3 letras:**

- `country_iso_code`: Código ISO de 3 letras (p.ej., 'ESP' para España)
- Útil para visualizaciones en Plotly
- 222 países

In [15]:
merged = real_estate.merge(gdp, on='country_norm', how='left')
print('Después de unir GDP:', len(merged), 'filas')

merged = merged.merge(gini, on='country_norm', how='left')
print('Después de unir Gini:', len(merged), 'filas')

merged = merged.merge(pop, on='country_norm', how='left')
print('Después de unir Población:', len(merged), 'filas')

merged = merged.merge(crime, on='country_norm', how='left')
print('Después de unir Crimen:', len(merged), 'filas')

merged = merged.merge(unemp, on='country_norm', how='left')
print('Después de unir Desempleo:', len(merged), 'filas')

Después de unir GDP: 147536 filas
Después de unir Gini: 147536 filas
Después de unir Población: 147536 filas
Después de unir Crimen: 147536 filas
Después de unir Desempleo: 147536 filas


**Unión secuencial de datasets:**

Usamos `merge(..., how='left')` para:
- Mantener todas las filas del dataset principal (inmuebles)
- Añadir información de cada país si existe
- Si un país no tiene cierta información, queda vacío (NaN)

Se verifica que no se pierdan filas en el proceso.

In [16]:
merged = merged.merge(lat[['country_norm', 'country_code', 'latitude', 'longitude']], on='country_norm', how='left')
print('Después de unir Ubicación:', len(merged), 'filas')

merged = merged.merge(codes[['country_norm', 'country_iso_code']], on='country_norm', how='left')
print('Después de unir Códigos ISO:', len(merged), 'filas')

Después de unir Ubicación: 147536 filas
Después de unir Códigos ISO: 147536 filas


**Unión final de datos geográficos:**

Añadimos:
- Coordenadas para mapas
- Códigos ISO para visualizaciones de Plotly

In [17]:
merged.to_csv('../../Data/merged_complete.csv', index=False, encoding='utf-8')
print('Archivo guardado: merged_complete.csv')

Archivo guardado: merged_complete.csv


**Guardado del archivo:**

- Se guarda el dataset completo en formato CSV
- Se usa `index=False` para no guardar índices
- Archivo resultante: `merged_complete.csv`

In [18]:
unmatched = merged[merged['gdp_ppp'].isna() & merged['country_norm'].notna()]['country']
unmatched = [str(x).strip() for x in unmatched.unique() if pd.notna(x)]

print('Resumen final:')
print('- Total de filas:', len(merged))
print('- Países únicos:', merged['country_norm'].nunique())
print('- Países sin datos de GDP:', unmatched)

Resumen final:
- Total de filas: 147536
- Países únicos: 28
- Países sin datos de GDP: ['Northern Cyprus']


**Validación final:**

Verifica qué países en el dataset de inmuebles no tuvieron coincidencia con datos macro:
- 147,536 propiedades
- 28 países representados
- Algunos países pueden no tener todos los datos (GDP, Gini, etc.)

In [19]:
print('\nMuestra del dataset final:')
print(merged.head())
print('\nColumnas disponibles:')
print(merged.columns.tolist())


Muestra del dataset final:
                                               title   country  \
0  2 room apartment 120 m² in Mediterranean Regio...    Turkey   
1              4 room villa 500 m² in Kalkan, Turkey    Turkey   
2          1 room apartment 65 m² in Antalya, Turkey    Turkey   
3             1 room apartment  in Pattaya, Thailand  Thailand   
4             2 room apartment  in Pattaya, Thailand  Thailand   

                                    location  building_construction_year  \
0               Mediterranean Region, Turkey                         NaN   
1  Kalkan, Mediterranean Region, Kas, Turkey                      2021.0   
2      Mediterranean Region, Antalya, Turkey                         NaN   
3      Chon Buri Province, Pattaya, Thailand                      2020.0   
4      Chon Buri Province, Pattaya, Thailand                      2026.0   

   building_total_floors  apartment_floor  apartment_rooms  \
0                    5.0              1.0              3

**Vista previa del resultado:**

Muestra las primeras filas y todas las columnas disponibles para verificar que la unión fue correcta.

**Próximos pasos:**
- Limpiar valores nulos
- Explorar correlaciones
- Visualizar en Streamlit o Power BI

## Limpieza de valores nulos

En este paso limpiamos nulos de forma simple:

1. Convertimos a numéricas las columnas que deberían ser numéricas.
2. Quitamos filas sin precio o sin país normalizado.
3. Rellenamos nulos numéricos con mediana por país y, si falta, mediana global.
4. Excepción: `building_total_floors` se rellena con moda por país (y moda global si falta) para mantener una variable discreta.
5. Rellenamos nulos de texto con `unknown`.

El resultado se guarda en `processed.csv` dentro de esta misma carpeta (`pre-procesamiento`).

In [35]:
# Copia de trabajo
cleaned = merged.copy()

# 1) Convertir columnas numéricas
numeric_cols = [
    'building_construction_year', 'building_total_floors', 'apartment_floor',
    'apartment_rooms', 'apartment_bedrooms', 'apartment_bathrooms',
    'price_in_USD', 'latitude', 'longitude', 'gini', 'gini_year',
    'crime_index', 'safety_index', 'unemployment_last', 'unemployment_previous'
]

for col in numeric_cols:
    if col in cleaned.columns:
        cleaned[col] = pd.to_numeric(cleaned[col], errors='coerce')

# Limpiar áreas (vienen como texto, por ejemplo: "120 m²")
for col in ['apartment_total_area', 'apartment_living_area']:
    if col in cleaned.columns:
        cleaned[col] = (
            cleaned[col]
            .astype(str)
            .str.replace('m²', '', regex=False)
            .str.replace('m2', '', regex=False)
            .str.extract(r'([0-9]+(?:\.[0-9]+)?)')[0]
        )
        cleaned[col] = pd.to_numeric(cleaned[col], errors='coerce')

# Limpiar numéricos macro en formato texto
if 'gdp_ppp' in cleaned.columns:
    cleaned['gdp_ppp'] = pd.to_numeric(cleaned['gdp_ppp'].astype(str).str.replace(',', '', regex=False), errors='coerce')

for col in ['population_2018', 'population_2019']:
    if col in cleaned.columns:
        cleaned[col] = pd.to_numeric(cleaned[col].astype(str).str.replace(',', '', regex=False), errors='coerce')

if 'population_change' in cleaned.columns:
    cleaned['population_change'] = pd.to_numeric(cleaned['population_change'].astype(str).str.replace('%', '', regex=False), errors='coerce')

# 2) Eliminar filas críticas sin precio o sin país
before_rows = len(cleaned)
cleaned = cleaned.dropna(subset=['country_norm', 'price_in_USD'])

# 3) Rellenar nulos numéricos
fill_numeric_cols = [
    'building_construction_year', 'building_total_floors', 'apartment_floor',
    'apartment_rooms', 'apartment_bedrooms', 'apartment_bathrooms',
    'apartment_total_area', 'apartment_living_area', 'gdp_ppp', 'gini',
    'crime_index', 'safety_index', 'unemployment_last', 'unemployment_previous',
    'population_2018', 'population_2019', 'population_change', 'latitude', 'longitude'
]

for col in fill_numeric_cols:
    if col in cleaned.columns:
        if col == 'building_total_floors':
            # Para plantas del edificio usamos moda (discreto), no mediana.
            def fill_mode_country(s):
                mode = s.mode(dropna=True)
                if not mode.empty:
                    return s.fillna(mode.iloc[0])
                return s

            cleaned[col] = cleaned.groupby('country_norm')[col].transform(fill_mode_country)

            global_mode = cleaned[col].mode(dropna=True)
            if not global_mode.empty:
                cleaned[col] = cleaned[col].fillna(global_mode.iloc[0])

            cleaned[col] = pd.to_numeric(cleaned[col], errors='coerce').round().astype('Int64')
        else:
            cleaned[col] = cleaned.groupby('country_norm')[col].transform(lambda s: s.fillna(s.median()))
            cleaned[col] = cleaned[col].fillna(cleaned[col].median())

# 4) Rellenar nulos de texto
fill_text_cols = ['country_code', 'country_iso_code', 'un_continent', 'un_statistical', 'unemployment_reference', 'unemployment_unit']
for col in fill_text_cols:
    if col in cleaned.columns:
        cleaned[col] = cleaned[col].fillna('unknown')

# Guardar resultado en la carpeta actual del notebook (no en Data)
output_path = 'processed.csv'
cleaned.to_csv(output_path, index=False, encoding='utf-8')

print('Limpieza de nulos completada')
print('Filas antes:', before_rows)
print('Filas después:', len(cleaned))
print('Archivo guardado:', output_path)

nulls_after = cleaned.isna().sum().sort_values(ascending=False)
print('\nTop 10 columnas con más nulos tras limpieza:')
print(nulls_after.head(10))

Limpieza de nulos completada
Filas antes: 147536
Filas después: 144961
Archivo guardado: processed.csv

Top 10 columnas con más nulos tras limpieza:
Country                       875
Country_y                     875
Country/Territory_x           875
Country_x                     875
Country/Territory_y           875
gini_year                     875
location                      131
country                       130
building_construction_year      0
title                           0
dtype: int64


## Vista rápida de `processed.csv`

Mostramos los primeros 20 registros del archivo procesado para validar nombres de columnas y contenido.

In [22]:
processed_preview = pd.read_csv('processed.csv')
print('Shape:', processed_preview.shape)

# Mostrar solo 8 filas (manteniendo todas las columnas)
display(processed_preview.head(8))

Shape: (144961, 38)


,title,country,location,building_construction_year,building_total_floors,apartment_floor,apartment_rooms,apartment_bedrooms,apartment_bathrooms,apartment_total_area,apartment_living_area,price_in_USD,image,url,country_norm,Country/Territory_x,gdp_ppp,Country_x,gini,gini_year,Country/Territory_y,un_continent,un_statistical,population_2018,population_2019,population_change,Country_y,crime_index,safety_index,Country,unemployment_last,unemployment_previous,unemployment_reference,unemployment_unit,country_code,latitude,longitude,country_iso_code
0,"2 room apartment 120 m² in Mediterranean Region, Turkey",Turkey,"Mediterranean Region, Turkey",2023.0,5.0,1.0,3.0,2.0,2.0,120.0,110.0,315209.0,https://realting.com/uploads/bigSlider/ab3/888989d8c45280f0e1e17d633ee94.webp,https://realting.com/property-for-sale/turkey/inest-homes/1766770,turkey,Turkey,2325617.0,Turkey,41.9,2018.0,Turkey,Asia,Western Asia,82340088.0,83429615.0,1.32,Turkey,39.50,60.50,Turkey,13.20,13.40,20-Aug,%,TR,38.963745,35.243322,TUR
1,"4 room villa 500 m² in Kalkan, Turkey",Turkey,"Kalkan, Mediterranean Region, Kas, Turkey",2021.0,2.0,3.0,3.0,2.0,1.0,500.0,480.0,1108667.0,https://realting.com/uploads/bigSlider/87b/67960b9229216decf200460d37dca.webp,https://realting.com/property-for-sale/turkey/inest-homes/1767446,turkey,Turkey,2325617.0,Turkey,41.9,2018.0,Turkey,Asia,Western Asia,82340088.0,83429615.0,1.32,Turkey,39.50,60.50,Turkey,13.20,13.40,20-Aug,%,TR,38.963745,35.243322,TUR
2,"1 room apartment 65 m² in Antalya, Turkey",Turkey,"Mediterranean Region, Antalya, Turkey",2023.0,5.0,2.0,2.0,1.0,1.0,65.0,60.0,173211.0,https://realting.com/uploads/bigSlider/030/a115091710bbd90676afae0ffb1df.webp,https://realting.com/property-for-sale/turkey/fsbo/1766776,turkey,Turkey,2325617.0,Turkey,41.9,2018.0,Turkey,Asia,Western Asia,82340088.0,83429615.0,1.32,Turkey,39.50,60.50,Turkey,13.20,13.40,20-Aug,%,TR,38.963745,35.243322,TUR
3,"1 room apartment in Pattaya, Thailand",Thailand,"Chon Buri Province, Pattaya, Thailand",2020.0,15.0,5.0,2.0,1.0,1.0,47.0,40.0,99900.0,https://realting.com/uploads/bigSlider/e9a/e061e09c8544dd3e449393b82e3b8.webp,https://realting.com/property-for-sale/thailand/alza-real-estate/1754337,thailand,Thailand,1338781.0,Thailand,36.4,2018.0,Thailand,Asia,South-eastern Asia,68863514.0,69037513.0,0.25,Thailand,40.01,59.99,Thailand,1.90,2.10,20-Aug,%,TH,15.870032,100.992541,THA
4,"2 room apartment in Pattaya, Thailand",Thailand,"Chon Buri Province, Pattaya, Thailand",2026.0,8.0,3.0,3.0,2.0,1.0,47.0,36.0,67000.0,https://realting.com/uploads/bigSlider/453/aa2f3e8c7387dd5f91d4271ecc4e0.webp,https://realting.com/property-for-sale/thailand/alza-real-estate/1754327,thailand,Thailand,1338781.0,Thailand,36.4,2018.0,Thailand,Asia,South-eastern Asia,68863514.0,69037513.0,0.25,Thailand,40.01,59.99,Thailand,1.90,2.10,20-Aug,%,TH,15.870032,100.992541,THA
5,"1 room apartment 28 m² in Batumi, Georgia",Georgia,"Abkhazia, Batumi, Georgia",2026.0,15.0,7.0,1.0,1.0,1.0,28.0,48.0,35622.0,https://realting.com/uploads/bigSlider/16f/406c7a33e86a7ca2f5285bf817e47.webp,https://realting.com/property-for-sale/georgia/flatiko-llc/1765932,georgia,Georgia,58174.0,Georgia,36.4,2018.0,Georgia,Asia,Western Asia,4002942.0,3996765.0,0.09,Georgia,20.50,79.50,Georgia,12.70,13.90,18-Dec,%,GE,42.315407,43.356892,GEO
6,"4 room apartment 245 m² in Yesiloez, Turkey",Turkey,"Yesiloez, Mediterranean Region, Alanya, Turkey",2007.0,2.0,3.0,5.0,4.0,4.0,245.0,245.0,274415.0,https://realting.com/uploads/bigSlider/9c8/6f1d74ccdc5e0d8153dbcbb49d147.webp,https://realting.com/property-for-sale/turkey/fsbo/1732620,turkey,Turkey,2325617.0,Turkey,41.9,2018.0,Turkey,Asia,Western Asia,82340088.0,83429615.0,1.32,Turkey,39.50,60.50,Turkey,13.20,13.40,20-Aug,%,TR,38.963745,35.243322,TUR
7,"1 room studio apartment in Pattaya, Thailand",Thailand,"Chon Buri Province, Pattaya, Thailand",2026.0,8.0,2.0,1.0,1.0,1.0,47.0,25.0,44700.0,https://realting.com/uploads/bigSlider/b50/f4bbb75591310bcd8fa4a68e2564e.webp,https://realting.com/pro

## Reordenar y limpiar columnas clave

En este paso vamos a:

1. Renombrar `title` a `apartament_m2`.
2. Convertir `price_in_USD` a euros.
3. Dejar `price_in_Euro` como columna visual (label formateada con `€`) y guardar su cálculo en `price_in_Euro_calculo`.
4. Formatear `apartment_total_area` y `apartment_living_area` como texto visual en `m²` (sin `.0`), y guardar sus versiones numéricas en `apartment_total_area_calculo` y `apartment_living_area_calculo`.
5. Reordenar columnas para que queden al inicio: `country`, `location`, `price_in_Euro`, `building_construction_year`, `apartament_m2`.
6. Mover al final las columnas de área visuales y las de cálculo.
7. Limpiar `location` quitando el país cuando se repite.
8. Limpiar `apartament_m2` quitando repeticiones de país y location (incluyendo repeticiones como `Yerevan, Yerevan`).
9. Quitar `.0` en columnas numéricas solo cuando el valor es entero real, manteniendo decimales útiles.
10. Guardar los cambios en `processed.csv` para TODO el CSV.

In [5]:
import re
import pandas as pd

df = pd.read_csv('processed.csv')

# Tipo de cambio editable (1 USD -> EUR)
USD_TO_EUR = 0.93


def split_terms(value):
    if pd.isna(value):
        return []
    text = str(value)
    parts = re.split(r'[,/;\-|]+', text)
    out = []
    seen = set()
    for part in parts:
        item = re.sub(r'\s+', ' ', part).strip(' .,-')
        if not item:
            continue
        key = item.casefold()
        if key not in seen:
            out.append(item)
            seen.add(key)
    return out


def dedupe_comma_parts(text):
    if pd.isna(text):
        return text
    parts = [p.strip() for p in str(text).split(',')]
    out = []
    seen = set()
    for p in parts:
        if not p:
            continue
        key = p.casefold()
        if key not in seen:
            out.append(p)
            seen.add(key)
    return ', '.join(out)


def remove_terms(text, terms):
    if pd.isna(text):
        return text
    value = str(text)
    for term in terms:
        term = str(term).strip()
        if not term:
            continue
        value = re.sub(rf'(?i)\b{re.escape(term)}\b', '', value)
    value = re.sub(r'\s*,\s*', ', ', value)
    value = re.sub(r'(,\s*){2,}', ', ', value)
    value = re.sub(r'\s+', ' ', value).strip(' ,.-')
    return dedupe_comma_parts(value)


def dedupe_repeated_words(text):
    if pd.isna(text):
        return text
    words = str(text).split()
    if not words:
        return text
    cleaned = [words[0]]
    for w in words[1:]:
        if w.casefold() != cleaned[-1].casefold():
            cleaned.append(w)
    return ' '.join(cleaned)


def remove_orphan_in(text):
    if pd.isna(text):
        return text
    value = str(text).strip()
    if re.fullmatch(r'(?i)in', value):
        return ''
    value = re.sub(r'(?i)\s+in\s*$', '', value).strip(' ,.-')
    value = re.sub(r'(?i)(^|,\s*)in(\s*,|$)', ' ', value)
    value = re.sub(r'\s*,\s*', ', ', value)
    value = re.sub(r'(,\s*){2,}', ', ', value)
    value = re.sub(r'\s+', ' ', value).strip(' ,.-')
    return value


def format_euro_es(value):
    if pd.isna(value):
        return ''
    num = float(value)
    sign = '-' if num < 0 else ''
    num = abs(num)
    if abs(num - round(num)) < 1e-9:
        integer_part = f"{int(round(num)):,}".replace(',', '.')
        return f"{sign}{integer_part} €"
    us = f"{num:,.2f}"
    es = us.replace(',', 'X').replace('.', ',').replace('X', '.')
    return f"{sign}{es} €"


def format_area_m2(value):
    if pd.isna(value):
        return ''

    num = float(value)
    if abs(num - round(num)) < 1e-9:
        return f"{int(round(num))} m²"

    text = f"{num:.2f}".rstrip('0').rstrip('.').replace('.', ',')
    return f"{text} m²"


def convert_integer_like_columns(frame):
    numeric_candidates = [
        'building_construction_year', 'building_total_floors', 'apartment_floor',
        'apartment_rooms', 'apartment_bedrooms', 'apartment_bathrooms',
        'gdp_ppp', 'gini_year', 'crime_index', 'safety_index',
        'population_2018', 'population_2019', 'apartment_total_area_calculo',
        'apartment_living_area_calculo'
    ]
    keep_decimals = {
        'latitude', 'longitude', 'gini', 'unemployment_last',
        'unemployment_previous', 'population_change', 'price_in_Euro_calculo'
    }
    converted = []
    for col in numeric_candidates:
        if col not in frame.columns or col in keep_decimals:
            continue
        s = pd.to_numeric(frame[col], errors='coerce')
        non_null = s.dropna()
        if non_null.empty:
            continue
        if ((non_null % 1).abs() < 1e-9).all():
            frame[col] = s.round().astype('Int64')
            converted.append(col)
        else:
            frame[col] = s
    return converted


# 0) Renombrar title -> apartament_m2
if 'title' in df.columns and 'apartament_m2' not in df.columns:
    df = df.rename(columns={'title': 'apartament_m2'})

# 0.1) Generar precio numérico en euros para cálculo
if 'price_in_USD' in df.columns:
    usd_numeric = pd.to_numeric(df['price_in_USD'], errors='coerce')
    df['price_in_Euro_calculo'] = (usd_numeric * USD_TO_EUR).round(2)
    df = df.drop(columns=['price_in_USD'])
elif 'price' in df.columns:
    price_numeric = pd.to_numeric(df['price'], errors='coerce')
    df['price_in_Euro_calculo'] = (price_numeric * USD_TO_EUR).round(2)
    df = df.drop(columns=['price'])
elif 'price_in_Euro_calculo' in df.columns:
    df['price_in_Euro_calculo'] = pd.to_numeric(df['price_in_Euro_calculo'], errors='coerce').round(2)
elif 'price_in_Euro' in df.columns:
    tmp = (
        df['price_in_Euro'].astype(str)
        .str.replace('€', '', regex=False)
        .str.strip()
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
    )
    df['price_in_Euro_calculo'] = pd.to_numeric(tmp, errors='coerce').round(2)

# 0.2) Columna visual de precio
if 'price_in_Euro_calculo' in df.columns:
    df['price_in_Euro'] = df['price_in_Euro_calculo'].apply(format_euro_es)

# 0.3) Preparar áreas: cálculo numérico + visual m²
if 'apartment_total_area' in df.columns:
    df['apartment_total_area_calculo'] = pd.to_numeric(df['apartment_total_area'], errors='coerce')
    df['apartment_total_area'] = df['apartment_total_area_calculo'].apply(format_area_m2)

if 'apartment_living_area' in df.columns:
    df['apartment_living_area_calculo'] = pd.to_numeric(df['apartment_living_area'], errors='coerce')
    df['apartment_living_area'] = df['apartment_living_area_calculo'].apply(format_area_m2)

# 1) Limpiar LOCATION quitando país repetido y duplicados.
if {'location', 'country'}.issubset(df.columns):
    df['location'] = df.apply(
        lambda row: remove_terms(row['location'], split_terms(row['country'])),
        axis=1
    )

# 2) Limpiar APARTAMENT_M2 quitando repeticiones de país/location y duplicados.
if 'apartament_m2' in df.columns:
    def clean_apartament(row):
        terms = []
        if 'country' in df.columns:
            terms.extend(split_terms(row.get('country')))
        if 'location' in df.columns:
            terms.extend(split_terms(row.get('location')))

        value = remove_terms(row['apartament_m2'], terms)
        value = dedupe_repeated_words(value)
        value = remove_orphan_in(value)

        loc = row.get('location', '')
        if isinstance(value, str) and isinstance(loc, str):
            if value.strip().casefold() == loc.strip().casefold():
                return ''

        return value

    df['apartament_m2'] = df.apply(clean_apartament, axis=1)

# 2.1) Quitar .0 en numéricas cuando no aporta información.
converted_cols = convert_integer_like_columns(df)

# 2.2) Forzar building_total_floors a entero si no hay decimales reales.
floor_decimal_count_before = 0
floor_converted_to_int = False
if 'building_total_floors' in df.columns:
    floors_numeric = pd.to_numeric(df['building_total_floors'], errors='coerce')
    floor_decimal_count_before = int(((floors_numeric.dropna() % 1).abs() >= 1e-9).sum())
    if floor_decimal_count_before == 0:
        df['building_total_floors'] = floors_numeric.round().astype('Int64')
        floor_converted_to_int = True
        if 'building_total_floors' not in converted_cols:
            converted_cols.append('building_total_floors')
    else:
        df['building_total_floors'] = floors_numeric

# 3) Reordenar columnas principales
priority = ['country', 'location', 'price_in_Euro', 'building_construction_year', 'apartament_m2']
priority = [c for c in priority if c in df.columns]

area_visual_tail = [c for c in ['apartment_total_area', 'apartment_living_area'] if c in df.columns]
calc_tail = [c for c in ['price_in_Euro_calculo', 'apartment_total_area_calculo', 'apartment_living_area_calculo'] if c in df.columns]

rest = [c for c in df.columns if c not in priority + area_visual_tail + calc_tail]
df = df[priority + rest + area_visual_tail + calc_tail]

# 4) Ordenar filas por país (A-Z) y reiniciar índice.
if 'country' in df.columns:
    df = df.sort_values(by='country', ascending=True, kind='stable')

df = df.reset_index(drop=True)

# 5) Guardar resultado para TODO el CSV.
df.to_csv('processed.csv', index=False, encoding='utf-8')

# Validaciones rápidas
repeated_count = 0
orphan_in_count = 0
year_decimal_count = 0
eur_null_count = 0
floor_decimal_count_after = 0
area_total_dot_zero_count = 0
area_living_dot_zero_count = 0

if {'location', 'apartament_m2'}.issubset(df.columns):
    repeated_count = (
        df['location'].fillna('').astype(str).str.strip().str.casefold()
        == df['apartament_m2'].fillna('').astype(str).str.strip().str.casefold()
    ).sum()

if 'apartament_m2' in df.columns:
    orphan_in_count = (
        df['apartament_m2'].fillna('').astype(str).str.match(r'(?i)^\s*in\s*$')
        | df['apartament_m2'].fillna('').astype(str).str.match(r'(?i)^.*\s+in\s*$')
    ).sum()

if 'building_construction_year' in df.columns:
    year_decimal_count = (
        pd.to_numeric(df['building_construction_year'], errors='coerce')
        .dropna()
        .astype(str)
        .str.contains(r'\.')
        .sum()
    )

if 'building_total_floors' in df.columns:
    floor_numeric_after = pd.to_numeric(df['building_total_floors'], errors='coerce')
    floor_decimal_count_after = int(((floor_numeric_after.dropna() % 1).abs() >= 1e-9).sum())

if 'price_in_Euro_calculo' in df.columns:
    eur_null_count = int(pd.to_numeric(df['price_in_Euro_calculo'], errors='coerce').isna().sum())

if 'apartment_total_area' in df.columns:
    area_total_dot_zero_count = int(df['apartment_total_area'].astype(str).str.contains(r'\.0\s*m²$', regex=True).sum())

if 'apartment_living_area' in df.columns:
    area_living_dot_zero_count = int(df['apartment_living_area'].astype(str).str.contains(r'\.0\s*m²$', regex=True).sum())

print('Tabla actualizada y guardada en processed.csv')
print('Tipo de cambio aplicado (USD->EUR):', USD_TO_EUR)
print('Filas totales afectadas:', len(df))
print('Primeras 6 columnas:', list(df.columns[:6]))
print('Últimas 5 columnas:', list(df.columns[-5:]))
print('Ejemplo precio mostrado:', df['price_in_Euro'].iloc[0] if 'price_in_Euro' in df.columns and len(df) > 0 else '')
print('Ejemplo area total mostrada:', df['apartment_total_area'].iloc[0] if 'apartment_total_area' in df.columns and len(df) > 0 else '')
print('Ejemplo area living mostrada:', df['apartment_living_area'].iloc[0] if 'apartment_living_area' in df.columns and len(df) > 0 else '')
print('Columnas convertidas a enteros (sin .0):', converted_cols)
print('Filas con location == apartament_m2:', int(repeated_count))
print("Filas con 'in' residual en apartament_m2:", int(orphan_in_count))
print('Filas con decimales en building_construction_year:', int(year_decimal_count))
print('Filas con parte decimal real en building_total_floors antes:', int(floor_decimal_count_before))
print('Filas con parte decimal real en building_total_floors después:', int(floor_decimal_count_after))
print('building_total_floors convertido a entero:', floor_converted_to_int)
print('Filas sin valor numérico en price_in_Euro_calculo:', int(eur_null_count))
print('Filas con ".0 m²" en apartment_total_area:', int(area_total_dot_zero_count))
print('Filas con ".0 m²" en apartment_living_area:', int(area_living_dot_zero_count))

Tabla actualizada y guardada en processed.csv
Tipo de cambio aplicado (USD->EUR): 0.93
Filas totales afectadas: 144961
Primeras 6 columnas: ['country', 'location', 'price_in_Euro', 'building_construction_year', 'apartament_m2', 'building_total_floors']
Últimas 5 columnas: ['apartment_total_area', 'apartment_living_area', 'price_in_Euro_calculo', 'apartment_total_area_calculo', 'apartment_living_area_calculo']
Ejemplo precio mostrado: 279.000 €
Ejemplo area total mostrada: 280 m²
Ejemplo area living mostrada: 86 m²
Columnas convertidas a enteros (sin .0): ['building_construction_year', 'building_total_floors', 'apartment_floor', 'apartment_rooms', 'apartment_bedrooms', 'apartment_bathrooms', 'gdp_ppp', 'gini_year', 'population_2018', 'population_2019']
Filas con location == apartament_m2: 0
Filas con 'in' residual en apartament_m2: 0
Filas con decimales en building_construction_year: 0
Filas con parte decimal real en building_total_floors antes: 0
Filas con parte decimal real en buildin

In [6]:
# Ajuste final de orden: apartment_total_area y apartment_living_area en posiciones 11 y 12
order_df = pd.read_csv('processed.csv')

area_cols = [c for c in ['apartment_total_area', 'apartment_living_area'] if c in order_df.columns]
all_cols = list(order_df.columns)

# Quitar temporalmente las columnas de área para recolocarlas
base_cols = [c for c in all_cols if c not in area_cols]

# Índice 10 = posición 11 (1-based)
insert_at = min(10, len(base_cols))
new_cols = base_cols[:insert_at] + area_cols + base_cols[insert_at:]

order_df = order_df[new_cols]
order_df.to_csv('processed.csv', index=False, encoding='utf-8')

pos_total = new_cols.index('apartment_total_area') + 1 if 'apartment_total_area' in new_cols else None
pos_living = new_cols.index('apartment_living_area') + 1 if 'apartment_living_area' in new_cols else None

print('Orden actualizado en processed.csv')
print('Posición apartment_total_area:', pos_total)
print('Posición apartment_living_area:', pos_living)
print('Primeras 15 columnas:', new_cols[:15])

Orden actualizado en processed.csv
Posición apartment_total_area: 11
Posición apartment_living_area: 12
Primeras 15 columnas: ['country', 'location', 'price_in_Euro', 'building_construction_year', 'apartament_m2', 'building_total_floors', 'apartment_floor', 'apartment_rooms', 'apartment_bedrooms', 'apartment_bathrooms', 'apartment_total_area', 'apartment_living_area', 'image', 'url', 'country_norm']


In [7]:
# Normalizar apartament_m2 para mostrar solo tipo de inmueble
import re
import pandas as pd

type_df = pd.read_csv('processed.csv')

def infer_property_type(text):
    if pd.isna(text):
        return 'unknown'

    value = str(text).strip().lower()
    if not value:
        return 'unknown'

    # Tipos ordenados por prioridad
    patterns = [
        ('penthouse', r'\bpenthouse\b'),
        ('duplex', r'\bduplex\b'),
        ('triplex', r'\btriplex\b'),
        ('townhouse', r'\btownhouse\b'),
        ('mansion', r'\bmansion\b'),
        ('villa', r'\bvilla\b'),
        ('chalet', r'\bchalet\b'),
        ('cottage', r'\bcottage\b'),
        ('loft', r'\bloft\b'),
        ('studio', r'\bstudio\b'),
        ('house', r'\bhouse\b'),
        ('apartment', r'\bapartment\b|\bflat\b|\bcondo\b'),
    ]

    for label, pattern in patterns:
        if re.search(pattern, value):
            return label

    # Fallback: si no detecta patrón claro, marcamos unknown
    return 'unknown'

if 'apartament_m2' in type_df.columns:
    type_df['apartament_m2'] = type_df['apartament_m2'].apply(infer_property_type)

# Guardar resultado
type_df.to_csv('processed.csv', index=False, encoding='utf-8')

print('apartament_m2 normalizada a tipo de inmueble')
print('Valores únicos (top 12):')
print(type_df['apartament_m2'].value_counts(dropna=False).head(12))

apartament_m2 normalizada a tipo de inmueble
Valores únicos (top 12):
apartament_m2
apartment    91972
house        31762
villa        10026
cottage       2884
duplex        2537
townhouse     2308
penthouse     1533
studio        1062
unknown        737
chalet         110
mansion         28
triplex          2
Name: count, dtype: int64


In [8]:
# Convertir URLs de imagen webp -> jpg para compatibilidad con Power BI
import re
import pandas as pd

img_df = pd.read_csv('processed.csv')

if 'image' in img_df.columns:
    original = img_df['image'].fillna('').astype(str)

    webp_mask = original.str.contains(r'(?i)\.webp($|\?)', regex=True)

    # Reemplaza solo extensión final conservando querystring si existe
    converted = original.str.replace(r'(?i)\.webp($|\?)', r'.jpg\1', regex=True)

    img_df['image'] = converted
    img_df.to_csv('processed.csv', index=False, encoding='utf-8')

    print('Columna image actualizada')
    print('URLs webp detectadas:', int(webp_mask.sum()))
    print('URLs convertidas a jpg:', int(webp_mask.sum()))
    print('Ejemplo antes:', original[webp_mask].head(1).tolist())
    print('Ejemplo después:', converted[webp_mask].head(1).tolist())
else:
    print('No existe la columna image en processed.csv')

C:\Users\Usuario\AppData\Local\Temp\ipykernel_10488\2128106257.py:10: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  webp_mask = original.str.contains(r'(?i)\.webp($|\?)', regex=True)


Columna image actualizada
URLs webp detectadas: 142691
URLs convertidas a jpg: 142691
Ejemplo antes: ['https://realting.com/uploads/bigSlider/493/0af3979d72edfd1d13e51c3323a69.webp']
Ejemplo después: ['https://realting.com/uploads/bigSlider/493/0af3979d72edfd1d13e51c3323a69.jpg']


In [9]:
# Enviar image y url al final de la tabla
end_df = pd.read_csv('processed.csv')

tail_cols = [c for c in ['image', 'url'] if c in end_df.columns]
base_cols = [c for c in end_df.columns if c not in tail_cols]

end_df = end_df[base_cols + tail_cols]
end_df.to_csv('processed.csv', index=False, encoding='utf-8')

print('Orden actualizado en processed.csv')
print('Últimas columnas:', list(end_df.columns[-5:]))

Orden actualizado en processed.csv
Últimas columnas: ['price_in_Euro_calculo', 'apartment_total_area_calculo', 'apartment_living_area_calculo', 'image', 'url']


In [10]:
# Renombrar country_norm y moverla al final para dejar claro su uso técnico de merge
norm_df = pd.read_csv('processed.csv')

old_col = 'country_norm'
new_col = 'country_norm_merge_key'

if old_col in norm_df.columns:
    norm_df = norm_df.rename(columns={old_col: new_col})

if new_col in norm_df.columns:
    cols = [c for c in norm_df.columns if c != new_col] + [new_col]
    norm_df = norm_df[cols]

norm_df.to_csv('processed.csv', index=False, encoding='utf-8')

print('Columna técnica de país actualizada')
print('Nombre final:', new_col if new_col in norm_df.columns else 'no encontrada')
print('Últimas columnas:', list(norm_df.columns[-5:]))

Columna técnica de país actualizada
Nombre final: country_norm_merge_key
Últimas columnas: ['apartment_total_area_calculo', 'apartment_living_area_calculo', 'image', 'url', 'country_norm_merge_key']


In [ ]:
# Cargar tabla legal final por pais y convertir ocupacion_legal -> riesgo_ocupacion
import pandas as pd

main_path = 'processed.csv'
main_df = pd.read_csv(main_path)

# Asegurar columnas objetivo
if 'riesgo_ocupacion' not in main_df.columns:
    main_df['riesgo_ocupacion'] = ''

if 'leyes_de_ocupacion_comentario' not in main_df.columns:
    main_df['leyes_de_ocupacion_comentario'] = ''

if 'leyes_de_ocupacion_url' not in main_df.columns:
    if 'ocupacion_fuente_url' in main_df.columns:
        main_df = main_df.rename(columns={'ocupacion_fuente_url': 'leyes_de_ocupacion_url'})
    else:
        main_df['leyes_de_ocupacion_url'] = ''

# Normalizar tipos para aceptar texto sin errores de dtype
for col in ['riesgo_ocupacion', 'leyes_de_ocupacion_comentario', 'leyes_de_ocupacion_url']:
    main_df[col] = main_df[col].fillna('').astype(str)

# Tabla enviada por el usuario (normalizada a nombres de pais del CSV)
country_updates = {
    'Armenia': {
        'riesgo': 'medio',
        'comentario': 'Propiedad privada protegida; ocupacion perseguida penalmente, aunque el tiempo de desalojo puede ser incierto. (Procedimiento mixto)',
        'fuente': ''
    },
    'Australia': {
        'riesgo': 'alto',
        'comentario': 'Existe la doctrina de posesion adversa, que permite reclamar la propiedad tras 12-15 anos de ocupacion continuada. (Procedimiento civil largo)',
        'fuente': 'Graham v. Nunn [2011] VSC 408; Morgan v. Taylor [2012] VSC 289'
    },
    'Austria': {
        'riesgo': 'bajo',
        'comentario': 'La ocupacion en si no es un delito penal, pero el allanamiento es punible. El desalojo puede ser muy rapido (en 24 horas). (Intervencion policial)',
        'fuente': '§ 134 StGB (Codigo Penal)'
    },
    'Belarus': {
        'riesgo': 'bajo',
        'comentario': 'La ocupacion ilegal de un terreno es una violacion grave de la legislacion. (Intervencion policial)',
        'fuente': 'Codigo de Tierras de la Republica de Belarus Nº 425-З'
    },
    'Croatia': {
        'riesgo': 'bajo',
        'comentario': 'La ocupacion no autorizada es ilegal. La constitucion protege la inviolabilidad de la propiedad. (Intervencion policial)',
        'fuente': ''
    },
    'Cyprus': {
        'riesgo': 'bajo',
        'comentario': 'Es un delito penal (Art. 281) con pena de hasta 5 anos de prision o multa de hasta 10.000 EUR. (Intervencion policial)',
        'fuente': ''
    },
    'Czech Republic': {
        'riesgo': 'bajo',
        'comentario': 'El allanamiento es un delito. (Intervencion policial)',
        'fuente': ''
    },
    'Finland': {
        'riesgo': 'bajo',
        'comentario': 'El allanamiento de morada es un delito. (Intervencion policial)',
        'fuente': ''
    },
    'Georgia': {
        'riesgo': 'bajo',
        'comentario': 'La ocupacion ilegal esta tipificada como delito. (Intervencion policial)',
        'fuente': 'Codigo Penal de Georgia'
    },
    'Greece': {
        'riesgo': 'bajo',
        'comentario': 'La ocupacion ilegal esta tipificada como delito en el Codigo Penal. (Intervencion policial)',
        'fuente': ''
    },
    'Hungary': {
        'riesgo': 'medio',
        'comentario': 'La ocupacion sin permiso es un delito penado con 1 a 5 anos de prision. (Procedimiento mixto)',
        'fuente': 'Act No. C of 2012 on the Penal Code, Art. 349'
    },
    'Indonesia': {
        'riesgo': 'bajo',
        'comentario': 'La ocupacion ilegal (allanamiento) es un delito. (Intervencion policial)',
        'fuente': ''
    },
    'Italy': {
        'riesgo': 'medio',
        'comentario': 'Delito penal (Art. 633 CP) con hasta 2 anos de prision, pero el proceso puede ser complejo. (Procedimiento mixto)',
        'fuente': ''
    },
    'Latvia': {
        'riesgo': 'bajo',
        'comentario': 'El fenomeno de la ocupacion es practicamente desconocido y esta criminalizado. (Intervencion policial)',
        'fuente': ''
    },
    'Lithuania': {
        'riesgo': 'bajo',
        'comentario': 'La ocupacion esta criminalizada. (Intervencion policial)',
        'fuente': ''
    },
    'Montenegro': {
        'riesgo': 'bajo',
        'comentario': 'La ocupacion es un acto ilegal. (Intervencion policial)',
        'fuente': ''
    },
    'Northern Cyprus': {
        'riesgo': 'bajo',
        'comentario': 'Es un delito penal con pena de hasta 5 anos de prision o multa de hasta 10.000 EUR. (Intervencion policial)',
        'fuente': ''
    },
    'Poland': {
        'riesgo': 'bajo',
        'comentario': 'La ocupacion es un acto ilegal. (Intervencion policial)',
        'fuente': ''
    },
    'Portugal': {
        'riesgo': 'bajo',
        'comentario': 'La ocupacion ilegal es un delito. La nueva ley refuerza la proteccion de la propiedad y agiliza la recuperacion de viviendas ocupadas. (Intervencion policial)',
        'fuente': 'Lei n.º 67/2025'
    },
    'Russia': {
        'riesgo': 'alto',
        'comentario': 'No existe un delito especifico de ocupacion. El desalojo es un proceso civil largo y complejo. (Procedimiento civil largo)',
        'fuente': ''
    },
    'Serbia': {
        'riesgo': 'medio',
        'comentario': 'La ocupacion ilegal de tierra ajena se castiga con multa o prision de hasta 3 anos. (Procedimiento mixto)',
        'fuente': 'Articulo 218 del Codigo Penal'
    },
    'Spain': {
        'riesgo': 'alto',
        'comentario': 'Delito penal (Art. 245 CP), pero el desalojo requiere un procedimiento civil largo. (Procedimiento civil largo)',
        'fuente': 'BOE Ley 5/2018'
    },
    'Thailand': {
        'riesgo': 'bajo',
        'comentario': 'Las leyes de propiedad son estrictas; la ocupacion es un delito con desalojos rapidos. (Intervencion policial)',
        'fuente': ''
    },
    'Turkey': {
        'riesgo': 'medio',
        'comentario': 'Contexto historico de gecekondu con amnistias; riesgo historicamente elevado. (Procedimiento mixto)',
        'fuente': ''
    },
    'UAE': {
        'riesgo': 'alto',
        'comentario': 'Existe la figura de la posesion adversa: se puede reclamar la propiedad tras 15 anos de posesion ininterrumpida. (Procedimiento civil largo)',
        'fuente': 'Articulo 1317 de la Ley Federal Nº'
    },
    'United States': {
        'riesgo': 'medio',
        'comentario': 'No hay ley federal; varia por estado. El allanamiento es ilegal, pero la posesion adversa permite reclamar la propiedad tras un periodo de 7 a 30 anos. (Procedimiento mixto)',
        'fuente': ''
    },
    'Uzbekistan': {
        'riesgo': 'bajo',
        'comentario': 'La ocupacion ilegal de tierras es un delito con multas y penas de prision. (Intervencion policial)',
        'fuente': 'Ley de 15 de noviembre de 2024'
    },
}

# Aplicar actualizacion por pais
updated_countries = []
for country, values in country_updates.items():
    mask = main_df['country'].astype(str).str.strip().eq(country)
    if mask.any():
        main_df.loc[mask, 'riesgo_ocupacion'] = values['riesgo']
        main_df.loc[mask, 'leyes_de_ocupacion_comentario'] = values['comentario']
        main_df.loc[mask, 'leyes_de_ocupacion_url'] = values['fuente']
        updated_countries.append(country)

# Si existe la columna vieja, retirarla para no duplicar conceptos
if 'ocupacion_legal' in main_df.columns:
    main_df = main_df.drop(columns=['ocupacion_legal'])

# Reordenar columnas nuevas justo despues de apartment_living_area
new_cols = [
    c for c in ['riesgo_ocupacion', 'leyes_de_ocupacion_comentario', 'leyes_de_ocupacion_url']
    if c in main_df.columns
]
cols = [c for c in main_df.columns if c not in new_cols]
insert_after = 'apartment_living_area'
idx = cols.index(insert_after) + 1 if insert_after in cols else len(cols)
main_df = main_df[cols[:idx] + new_cols + cols[idx:]]

# Guardar resultado
main_df.to_csv(main_path, index=False, encoding='utf-8')

print('CSV principal actualizado con tabla legal final')
print('Paises actualizados:', len(updated_countries))
print('Sin actualizar:', sorted(set(main_df['country'].dropna().astype(str).str.strip().unique()) - set(updated_countries)))
print('Distribucion de riesgo:')
print(main_df['riesgo_ocupacion'].value_counts(dropna=False))
print('Posiciones finales:')
for c in ['apartment_living_area', 'riesgo_ocupacion', 'leyes_de_ocupacion_comentario', 'leyes_de_ocupacion_url']:
    if c in main_df.columns:
        print('-', c, main_df.columns.get_loc(c) + 1)

TypeError: Invalid value '' for dtype 'float64'